# FruitCount — Pipeline Kaggle lấy source từ GitHub

Giữ nguyên pipeline chuẩn bị dữ liệu, huấn luyện, resume checkpoint, đánh giá và so sánh của kaggle_pipeline.ipynb.

Trước khi chạy:
1. Commit và push source lên nhánh TLuan của repository GitHub.
2. Bật Internet và GPU trong Settings của notebook Kaggle.
3. Add Input bộ MinneApple và sửa đường dẫn ảnh/mask ở cell cấu hình dữ liệu.

Source được clone vào /kaggle/working/FruitCountGit. Không cần upload source thành Kaggle Dataset.
Checkpoint được lưu vào /kaggle/working/runs sau mỗi epoch. Khi đổi phiên, cần lưu output phiên trước và khôi phục runs để resume.
Clone đã có được dùng lại, không tự pull. Muốn lấy code mới, dùng phiên mới hoặc đổi WORK sang thư mục chưa tồn tại.


In [7]:
import os, shutil, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/QNhut/BTL_CV.git'
BRANCH = 'TLuan'
WORK = '/kaggle/working/FruitCountGit'

work_dir = Path(WORK)
if not work_dir.exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, WORK],
        check=True,
    )
else:
    if not (work_dir / '.git').is_dir():
        raise RuntimeError(f'{WORK} exists but is not a Git checkout. Choose another WORK directory.')
    current_branch = subprocess.check_output(
        ['git', '-C', WORK, 'branch', '--show-current'], text=True
    ).strip()
    if current_branch != BRANCH:
        raise RuntimeError(f'Existing checkout uses {current_branch}, expected {BRANCH}. Choose another WORK directory.')

for required in ('src', 'scripts/train.py', 'configs/dataset.yaml', 'requirements.txt'):
    if not (work_dir / required).exists():
        raise FileNotFoundError(f'Missing project file: {required}')

os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)
print('Source:', REPO_URL, '| branch:', BRANCH)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-c', "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"], check=True)


Source: https://github.com/QNhut/BTL_CV.git | branch: TLuan
cbee25c
torch 2.11.0+cu128 | cuda True


CompletedProcess(args=['/usr/bin/python3', '-c', "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"], returncode=0)

## 1. Cấu hình dữ liệu
Sửa đường dẫn cho đúng cấu trúc bộ dữ liệu bạn Add vào (xem `/kaggle/input`).

In [8]:
from src.utils import load_yaml, save_yaml
from pathlib import Path

ds = load_yaml('configs/dataset.yaml')
# Set both paths explicitly if more than one dataset is attached.
IMAGES_DIR = None
MASKS_DIR = None
if IMAGES_DIR is None or MASKS_DIR is None:
    candidates = [p for p in Path('/kaggle/input').rglob('images')
                  if p.is_dir() and (p.parent / 'masks').is_dir()
                  and 'test' not in p.parts and 'val' not in p.parts]
    train_candidates = [p for p in candidates if 'train' in p.parts]
    candidates = train_candidates or candidates
    if len(candidates) != 1:
        raise ValueError(f'Set IMAGES_DIR and MASKS_DIR explicitly. Found image/mask pairs: {candidates}')
    IMAGES_DIR = str(candidates[0])
    MASKS_DIR = str(candidates[0].parent / 'masks')
for folder in (IMAGES_DIR, MASKS_DIR):
    if not Path(folder).is_dir():
        raise FileNotFoundError(f'Dataset directory does not exist: {folder}')
ds['paths']['images_dir'] = IMAGES_DIR
ds['paths']['masks_dir'] = MASKS_DIR
ds['processed_dir'] = '/kaggle/working/processed'
save_yaml(ds, 'configs/dataset.yaml')
print(ds['paths'])


{'images_dir': '/kaggle/input/datasets/nhtnguye/dataminesapple/detection/detection/train/images', 'masks_dir': '/kaggle/input/datasets/nhtnguye/dataminesapple/detection/detection/train/masks', 'mask_suffix': '', 'labels_dir': None, 'coco_json': None}


In [ ]:
import subprocess
import shlex
import sys

def run_python(arguments):
    subprocess.run([sys.executable, *shlex.split(arguments)], check=True)

run_python('scripts/prepare_data.py')


13:34:10 | train  470 ảnh | quả/ảnh: min=1  TB=42.2  max=121 | tổng=19837
13:34:10 | val    100 ảnh | quả/ảnh: min=1  TB=41.8  max=123 | tổng=4180
13:34:10 | test   100 ảnh | quả/ảnh: min=3  TB=41.6  max=117 | tổng=4159


: 

## 2. Huấn luyện
Kaggle giới hạn ~9–12 giờ/phiên và có thể ngắt; `--resume` tiếp tục từ `last.pt`. Chạy từng model ở các cell/phiên riêng nếu cần.

In [ ]:
MODELS = ['yolov8', 'csrnet', 'yolo11', 'faster_rcnn', 'mask_rcnn']          # thêm: 'yolo11', 'faster_rcnn', 'mask_rcnn'
EPOCHS = 40                           # ví dụ 5 để chạy thử nhanh

from pathlib import Path

# Checkpoints are saved after each epoch by the training code.
# To resume in a new Kaggle session, restore previous runs here first.
RUNS = Path('/kaggle/working/runs')
for m in MODELS:
    cfg = load_yaml(f'configs/models/{m}.yaml')
    checkpoint = RUNS / cfg['name'] / 'weights' / 'last.pt'
    resume = '--resume' if checkpoint.is_file() else ''
    extra = f'--epochs {EPOCHS}' if EPOCHS else ''
    print(f'{m}: resume {checkpoint}' if resume else f'{m}: start training')
    run_python(f'scripts/train.py --model configs/models/{m}.yaml --out {RUNS} {extra} {resume}')


yolov8: start training
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
13:22:29 | Chưa có dataset YOLO, đang xuất từ records...
Traceback (most recent call last):
  File "/kaggle/working/FruitCountGit/scripts/train.py", line 78, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/FruitCountGit/scripts/train.py", line 59, in main
    by_split = {s: load_split_records(processed, s) for s in ("train", "val", "test")}
                   ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/kaggle/working/FruitCountGit/src/data/splits.py", line 73, in load_split_records
    records = load_records(processed_dir / "records.json")
  File "/kaggle/working/FruitCountGit/src/data/annotations.py", line 233, in load_records
    return [ImageRecord.from_dict(d) for

## 3. Đánh giá trên tập test

In [ ]:
for m in MODELS:
    run_python(f'scripts/evaluate.py --model configs/models/{m}.yaml --out /kaggle/working/runs --split test --speed')

# YOLO + tiling (SAHI-style)
run_python(f'scripts/evaluate.py --model configs/models/yolov8.yaml --out /kaggle/working/runs --split test --tile --tag _tile')


Traceback (most recent call last):
  File "/kaggle/working/FruitCountGit/scripts/evaluate.py", line 76, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/FruitCountGit/scripts/evaluate.py", line 44, in main
    pred.model.load(str(weights))
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/kaggle/working/FruitCountGit/src/models/yolo.py", line 86, in load
    self.model = YOLO(str(weights))
                 ~~~~^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/ultralytics/models/yolo/model.py", line 84, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/ultralytics/engine/model.py", line 138, in __init__
    self._load(model, task=task)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/ultralytics/engine/model.py", line 253, in _load
    self.model, self.ckpt = load_checkpoint(weights)
                         

In [ ]:
run_python(f'scripts/compare.py --runs /kaggle/working/runs --split test')

import pandas as pd
pd.read_csv('/kaggle/working/runs/comparison/comparison_test.csv')

Chưa có metrics_test*.json trong /kaggle/working/runs; hãy chạy scripts/evaluate.py trước.


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/runs/comparison/comparison_test.csv'

## 4. Phân tích: quả chồng lấn nhiều thì model nào đếm tốt hơn?

In [ ]:
from IPython.display import Image, display
display(Image('/kaggle/working/runs/comparison/error_by_count_test.png'))
display(Image('/kaggle/working/runs/comparison/comparison_test.png'))

## 5. Ví dụ minh họa (cho báo cáo)

In [ ]:
import glob, random
from src.data import load_split_records

test = load_split_records('/kaggle/working/processed', 'test')
samples = random.Random(0).sample(test, min(4, len(test)))
os.makedirs('/kaggle/working/samples', exist_ok=True)
for r in samples:
    shutil.copy(r.image_path, f'/kaggle/working/samples/{r.image_id}.png')
    print(r.image_id, 'GT =', r.count)

for m in MODELS:
    run_python(f'scripts/predict.py --model configs/models/{m}.yaml --runs /kaggle/working/runs --source /kaggle/working/samples --out /kaggle/working/vis/{m}')


In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg

ids = [r.image_id for r in samples]
fig, axes = plt.subplots(len(ids), len(MODELS), figsize=(6 * len(MODELS), 4 * len(ids)), squeeze=False)
for i, iid in enumerate(ids):
    for j, m in enumerate(MODELS):
        axes[i][j].imshow(mpimg.imread(f'/kaggle/working/vis/{m}/{iid}_count.jpg'))
        axes[i][j].set_title(m); axes[i][j].axis('off')
plt.tight_layout(); plt.show()